# Lab 3: Binary Classification & Cost-Sensitive Thresholding

**Classification Framing:** Predicting `escalated` ($1 =$ escalated, $0 =$ not escalated) is a **supervised binary classification** task because the target is discrete with two mutually exclusive classes. The **positive class ($1$)** represents an **escalated ticket**, which carries higher operational risk when missed than a non-escalated ticket routed for review.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay
)

df = pd.read_csv("classification_escalation.csv")
print("Shape:", df.shape)
display(df.head())
print(df.info())
print("\nMissing values:\n", df.isna().sum())
print("\nTarget prevalence:\n", df["escalated"].value_counts(normalize=True).rename("proportion"))

In [2]:
numeric_features = ["complexity_score", "team_experience_years", "customer_tenure_months"]
categorical_features = ["priority", "channel"]
target_col = "escalated"

X = df[numeric_features + categorical_features]
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Majority Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
    "k-NN (k=15)": KNeighborsClassifier(n_neighbors=15),
}

fitted = {}
comp_rows = []
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None
    comp_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    })

comp_df = pd.DataFrame(comp_rows).set_index("Model")
display(comp_df.round(4))

In [3]:
lr_pipe = fitted["Logistic Regression"]
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1]
y_pred_lr_05 = (y_proba_lr >= 0.50).astype(int)

print(classification_report(y_test, y_pred_lr_05, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr_05, display_labels=["Not Escalated (0)", "Escalated (1)"], ax=axes[0], cmap="Blues"
)
axes[0].set_title("Confusion Matrix — Logistic Regression (t = 0.50)")

RocCurveDisplay.from_predictions(y_test, y_proba_lr, name="Logistic Regression", ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance (AUC = 0.50)")
axes[1].set_title("ROC Curve — Logistic Regression")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

In [4]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
t_rows = []
for t in thresholds:
    pred_t = (y_proba_lr >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    cost = 5 * fn + 1 * fp
    t_rows.append({
        "Threshold": t,
        "Precision": precision_score(y_test, pred_t, zero_division=0),
        "Recall": recall_score(y_test, pred_t, zero_division=0),
        "F1": f1_score(y_test, pred_t, zero_division=0),
        "FP": fp,
        "FN": fn,
        "Cost (5*FN + 1*FP)": cost,
    })

t_df = pd.DataFrame(t_rows)
display(t_df.round({"Precision": 4, "Recall": 4, "F1": 4}))

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", KNeighborsClassifier())])
grid = GridSearchCV(
    knn_pipe,
    param_grid={"classifier__n_neighbors": [5, 9, 15, 21, 31]},
    scoring="f1",
    cv=cv,
    return_train_score=True,
)
grid.fit(X_train, y_train)

cv_df = pd.DataFrame(grid.cv_results_)[
    ["param_classifier__n_neighbors", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]
].rename(columns={"param_classifier__n_neighbors": "k", "mean_test_score": "mean_cv_f1"})
display(cv_df.sort_values("k").round(4))
print("Best k:", grid.best_params_["classifier__n_neighbors"], "| Best 5-fold CV F1:", round(grid.best_score_, 4))

## Recommendation

For escalating service tickets, I recommend deploying Logistic Regression paired with a cost-sensitive probability threshold rather than relying on the default 0.50 cutoff. In our dataset of 900 tickets (20.67% escalated), a majority-class baseline achieves 79.56% accuracy on the test set while scoring 0.00 across precision, recall, and F1. Among the evaluated models at threshold 0.50, Logistic Regression and Linear Discriminant Analysis tie on accuracy (0.8222), precision (0.7500), recall (0.1957), and F1 (0.3103), while Logistic Regression achieves the strongest ROC-AUC (0.8111 vs. 0.8090 for LDA and 0.7046 for k-NN at k=15). Even tuning k-NN via stratified 5-fold cross-validation selects k=5 with a mean CV F1 of only 0.2433 and test F1 of 0.2034.

Under the cost function 5*FN + 1*FP, missing an escalation (False Negative) is five times as costly as flagging a non-escalated ticket for review (False Positive). Missed escalations risk SLA breaches, prolonged customer outages, churn, or compliance issues, whereas False Positives primarily consume triage capacity. Lowering the threshold from 0.50 to 0.20 raises Logistic Regression recall from 0.1957 to 0.7826 (cutting FN from 37 to 10 while increasing FP from 3 to 48) and reduces test cost from 188 down to 98—the lowest cost among tested cutoffs. Because selecting the minimum-cost threshold directly on the test set introduces selection optimism, thresholds should be validated using out-of-fold predictions and theoretically grounded via p* = C_FP / (C_FP + C_FN) = 1 / 6 ≈ 0.1667.

Important limitations remain. The dataset is synthetic and contains missing values in team_experience_years (31), customer_tenure_months (22), and priority (18). Real-world ticket logs experience temporal drift, noisy human priority labels, and potentially non-random missingness, while linear decision boundaries may miss non-linear feature interactions.

Responsible deployment requires guardrails against alert fatigue and bias. Setting a low threshold increases False Positives (48 at t=0.20, precision 0.4286), which can overload support engineers and delay urgent tickets if volume spikes. Furthermore, historical escalation patterns across channel or priority may encode operational biases. Before production rollout, we should evaluate probability calibration, run shadow testing on live tickets, monitor drift and false-positive workload, and retain human-in-the-loop triage.